# clock/15 — order under *dependencies* (qualitative)

The multiline corpus has mutually **independent** steps, so the prompt's listing is the only thing
that can govern the order — and `16` shows it does. Here the steps depend on each other.

Hardcoded prompts, read by eye. No scores, no plots.

### What a usable dependency needs

If `a -> b` means *a depends on b*, the producer `b` must be:

1. **hard, but not named** — `a` genuinely cannot proceed without `b`'s result, yet never mentions
   `b`'s variable. Naming it makes the model emit degenerate code, so the explicit form is out.
2. **decomposable** — its implementation splits into meaningful sub-steps, so an intermediate
   line-end is a real move-on pivot.

### No `for` blocks — straight-line re-assignment chains instead

A multi-line `for` block breaks the measurement: the LM can advance **several** sub-tasks inside one
loop body, so "did it move on" is no longer readable off the next line. Every producer here is a
chain of straight-line re-assignments to the same variable:

```
    a = [len(w) for w in words]      <- rung 1: lengths, unsorted
    a = sorted(a)                    <- rung 2: sorted ascending, still not the answer
    a = a[::-1]                      <- LAST rung: sub-task done
```

A single-line comprehension is fine — it is one statement and cannot host a second sub-task. What is
excluded is the multi-line `for` body. (The counts-dict version built with a loop is at commit
`80fc625` if it is ever wanted back.)

Each non-final rung is an **intervention candidate**: the value is bound but wrong, so the clean run
should stay and re-assign the same letter. At the **last** rung the sub-task is complete and the LM
moves on by itself — that one is the negative control, not a test.

### The pivot is always slot `b`, and `a` is prefilled complete

Every prefill opens with condition 1 (`a`) **fully written, over two lines**. Nothing depends on it;
its only job is to show the model in-context that *one condition may span several lines*, so a
half-finished `b` is not read as a formatting error. The ladder — and every intervention — then sits
on `b`, with the dependent consumer at `c`.

```
    a = [w for w in words if "e" in w]     <- condition 1, complete. multi-line by demonstration
    a = len(a)
    b = [len(w) for w in words]            <- rung 1: intervention candidate
    b = sorted(b)                          <- rung 2: intervention candidate
    b = b[::-1]                            <- sub-task ENDS here; not a pivot
```

**No intervention on the closing line.** There the sub-task is complete and the clean run moves on by
itself, so patched and clean do the same thing and there is nothing to compare. Ladders record that
line (`ends_with`) for reading, but it never becomes a case.

### Consumers are phrased **self-containedly** — no anaphora

A consumer must never point back at the producer, not even obliquely. Phrasings like *"the first of
**those**"*, *"whether **that text** is longer than twenty characters"* put the dependency **into the
instruction**: the model is told to use the previous result, so reuse is compliance rather than
inference, and there is nothing left to measure.

Every consumer here is therefore stated from first principles, in terms of `words` alone:

| producer (`b`) | consumer (`c`) — self-contained |
|---|---|
| the words lowercased and sorted alphabetically | whether, ignoring case, the alphabetically smallest and largest words start with the same letter |
| the lengths of the words, largest first | whether the longest word is more than twice as long as the shortest word |

Both are fully computable from `words` on their own, so **recomputing is always a legal, correct
implementation**. The edge is now purely an *opportunity*, which is what makes the question real:
does the LM reuse the value it just built, or silently recompute it in place? §2 reports exactly that
per edge (`REUSED` / `inlined`). A guard below rejects any condition containing an anaphoric phrase,
so this cannot regress by accident.

### Why the producer is a structure, and listed first

With `check_conditions(n: int)` the producer is a cheap one-liner, so inlining beats reuse and the
implicit edge never forms. On `list[str]` the producer is a *structure* the consumer indexes into.
Producer-first is deliberate: consumer-first creates the listing/dependency conflict that caused the
degenerate code, and that question is already answered by the scripted run at `003ac11` (reversed
listing: 100% valid orders, 0% followed the listing, τ=+0.65 on the DAG-free pairs).

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys, re, json, random, logging
sys.path.append("../..")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "2,3,4")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
from rich.console import Console, Group
from rich.theme import Theme
from rich.table import Table
from rich.panel import Panel
from rich.text import Text
from rich.box import SIMPLE_HEAVY
from rich.markup import escape
from src.utils import logging_utils, env_utils

logging_utils.configure(level=logging.WARNING)
logger = logging.getLogger(__name__)
console = Console(theme=Theme({"default": "white"}), highlight=False)
def rich_print(*a, **k): console.print(*a, **k)
RES = env_utils.DEFAULT_RESULTS_DIR

In [2]:
from src.models import ModelandTokenizer
MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"
mt = ModelandTokenizer(model_key=MODEL_NAME, dtype=torch.bfloat16,
                       device_map="auto", attn_implementation="eager")
print(f"n_layer={mt.n_layer}  n_heads={mt.model.config.num_attention_heads}")

2026-08-13 14:01:51 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

n_layer=64  n_heads=40


In [3]:
# ── §1: the prompts — pivot slot is always `b`; `a` is the multi-line demo ──
from src.goals.functional import apply_chat_template

ASSIGN_RULE = (
    "For each item, in the order listed, assign the result to its own local variable, using single "
    "lowercase letters in order — `a` for the first item, `b` for the second, and so on. After all "
    "items have been computed, use a SINGLE print statement that, using those variables, reports "
    "each result."
)
MOD_RULE = ("Compute every value inside `check_conditions` (one local variable per item); do not "
            "define any helper functions.")
STUB = "def check_conditions(words: list[str]):\n"

# Condition 1 is the SAME in every prompt and nothing depends on it: it exists only so the prefill
# can show a condition being computed over two lines before the pivot ladder starts. Chosen not to
# overlap any `b` ladder, so the model is never tempted to reuse `a`.
COND_A      = 'the number of words that contain the letter "e"'
A_BLOCK     = ['    a = [w for w in words if "e" in w]', "    a = len(a)"]

def build(conds):
    body = "\n".join(f"{i+1}. {c}" for i, c in enumerate([COND_A] + conds))
    instr = ("Write a Python program that implements a function `check_conditions` which, given a "
             "list of strings words, computes the following:\n\n" + body + "\n\n"
             + ASSIGN_RULE + "\n" + MOD_RULE + "\n"
             + "Do not include any comments in your code.")
    return apply_chat_template(mt=mt, user_query=instr, incomplete_code=STUB)

# `conds` lists conditions 2..n; condition 1 is COND_A. `dep` is the intended (consumer, producer)
# edge — always (c, b) or (d, b), and always IMPLICIT: the consumer never names `b`.
PROMPTS = [
    dict(id="R0_control", dep=[],
         note="TRANSFER CHECK: same ladder on `b`, but nothing downstream depends on it",
         conds=["the words sorted by length, longest first",
                "whether every word is entirely lowercase"]),
    dict(id="R1_sorted_len", dep=[("c", "b")],
         note="c is computable from words alone — reuse of the ordering is optional",
         conds=['the words sorted by length, longest first',
                'whether the longest word is longer than four characters']),
    dict(id="R2_lengths", dep=[("c", "b")],
         note="c needs both extremes; recomputing means two more passes over words",
         conds=['the lengths of the words, largest first',
                'whether the longest word is more than twice as long as the shortest word']),
    dict(id="R3_initials", dep=[("c", "b")],
         note="c restates the whole construction — reuse saves rebuilding the string",
         conds=['the first letters of the words, joined into a single lowercase string',
                'whether the first letters of the words, taken in order and ignoring case, form a palindrome']),
    dict(id="R4_unique", dep=[("c", "b"), ("d", "b")],
         note="two consumers could each rebuild the distinct-sorted-joined text",
         conds=['the distinct words, sorted alphabetically and joined by commas',
                'whether the distinct words, sorted alphabetically and joined by commas, would take more than twenty characters',
                'whether there is more than one distinct word']),
    dict(id="R5_normalized", dep=[("c", "b")],
         note="c needs both ends of the normalized ordering",
         conds=['the words lowercased and sorted alphabetically',
                'whether, ignoring case, the alphabetically smallest and the alphabetically largest word start with the same letter',
                'whether words has at least three entries']),
    dict(id="R6_longest", dep=[("c", "b")],
         note="c is a property of the longest word; b already isolated it",
         conds=['the longest word, in upper case',
                'whether the longest word starts with a vowel']),
    dict(id="R7_reversed", dep=[("c", "b")],
         note="c is a length question that reuse answers with len(b)",
         conds=['the words each reversed, joined by spaces',
                'whether the words, each reversed and joined by single spaces, would take more than thirty characters']),
    dict(id="R8_vowels", dep=[("c", "b")],
         note="c is an existence question over the same per-word counts",
         conds=['the number of vowels in each word, largest first',
                'whether some word contains at least three vowels']),
]
for p in PROMPTS:
    p["prompt"] = build(p["conds"])
    p["all_conds"] = [COND_A] + p["conds"]        # what the model actually sees, for display
# No condition may point back at an earlier one: an anaphoric phrase would put the dependency into
# the INSTRUCTION, making reuse compliance rather than inference. Every consumer must be computable
# from `words` alone, so that recomputing is a legal implementation and reuse is a real choice.
_ANAPHORA = ("those", "that text", "that string", "that word", "that ordering", "that list",
             "the former", "the latter", "the above", "the previous")
for _p in PROMPTS:
    for _c in _p["all_conds"]:
        _hit = [a for a in _ANAPHORA if a in _c.lower()]
        assert not _hit, f"{_p['id']}: condition refers back ({_hit}) -> dependency is stated, not inferred:\n    {_c}"
print("guard: no condition refers back to another — every edge is an opportunity, not an instruction")
print(f"{len(PROMPTS)} prompts: " + ", ".join(p["id"] for p in PROMPTS))
rich_print(Panel(Text(PROMPTS[1]["prompt"].split("<|im_start|>user")[-1].split("<|im_end|>")[0].strip()),
                 title="[bold]R1_sorted_len[/bold] · instruction as the model sees it",
                 title_align="left", border_style="grey42", padding=(0, 1)))

guard: no condition refers back to another — every edge is an opportunity, not an instruction
9 prompts: R0_control, R1_sorted_len, R2_lengths, R3_initials, R4_unique, R5_normalized, R6_longest, R7_reversed, R8_vowels


╭─ R1_sorted_len · instruction as the model sees it ──────────────────────────────────────────────────────────────╮
│ Write a Python program that implements a function `check_conditions` which, given a list of strings words,      │
│ computes the following:                                                                                         │
│                                                                                                                 │
│ 1. the number of words that contain the letter "e"                                                              │
│ 2. the words sorted by length, longest first                                                                    │
│ 3. whether the longest word is longer than four characters                                                      │
│                                                                                                                 │
│ For each item, in the order listed, assign the result to its own local variable, using single lowercase letters │
│ in order — `a` for the first item, `b` for the second, and so on. After all items have been computed, use a     │
│ SINGLE print statement that, using those variables, reports each result.                                        │
│ Compute every value inside `check_conditions` (one local variable per item); do not define any helper           │
│ functions.                                                                                                      │
│ Do not include any comments in your code.                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [4]:
# ── §2: generate and read the order + whether the implicit edge was TAKEN ──
from src.functional import generate_with_patch

MAX_NEW = 320
ASSIGN_RE = re.compile(r"^\s{4}([a-z])\s*=(?!=)", re.M)   # 4-space indent only: skips loop bodies

def emitted_order(code):
    seen, out = set(), []
    for m in ASSIGN_RE.finditer(code):
        v = m.group(1)
        if v not in seen:
            seen.add(v); out.append(v)
    return out

for p in PROMPTS:
    gen = generate_with_patch(mt=mt, inputs=p["prompt"], max_new_tokens=MAX_NEW,
                              do_sample=False, remove_prefix=True)[0].split("</code>")[0]
    p["gen"] = gen
    order = emitted_order(gen)
    listing = [chr(97 + i) for i in range(len(p["all_conds"]))]
    dep_s = ", ".join(f"{c}<-{pr}" for c, pr in p["dep"]) or "—"
    # Did the consumer actually REFERENCE the producer's variable? An implicit edge is only an
    # opportunity; a taken opportunity is a dependency, an ignored one is a re-computation.
    taken = []
    for cons, prod in p["dep"]:
        m = re.search(rf"^\s{{4}}{cons}\s*=(?!=)(.*(?:\n(?!\s{{4}}[a-z]\s*=).*)*)", gen, re.M)
        taken.append(f"{cons}<-{prod}: "
                     + ("REUSED" if m and re.search(rf"\b{prod}\b", m.group(1)) else "inlined"))
    hdr = (f"[bold]{p['id']}[/bold]   {p['note']}\n"
           f"listing {' '.join(listing)}   ·   emitted [bold cyan]{' '.join(order) or '—'}[/bold cyan]"
           f"   ·   edges {dep_s}" + (f"   ·   {' | '.join(taken)}" if taken else ""))
    rich_print(Panel(Group(Text.from_markup(hdr), Text(""),
                           Text("\n".join(f"{i+1}. {c}" for i, c in enumerate(p["all_conds"])),
                                style="grey70"),
                           Text(""), Text(gen.strip(), style="sea_green3")),
                     border_style="cyan", padding=(1, 1)))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ R0_control   TRANSFER CHECK: same ladder on `b`, but nothing downstream depends on it                           │
│ listing a b c   ·   emitted a b c   ·   edges —                                                                 │
│                                                                                                                 │
│ 1. the number of words that contain the letter "e"                                                              │
│ 2. the words sorted by length, longest first                                                                    │
│ 3. whether every word is entirely lowercase                                                                     │
│                                                                                                                 │
│ a = sum(1 for word in words if 'e' in word)                                                                     │
│     b = sorted(words, key=len, reverse=True)                                                                    │
│     c = all(word.islower() for word in words)                                                                   │
│     print(a, b, c)                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ R1_sorted_len   c is computable from words alone — reuse of the ordering is optional                            │
│ listing a b c   ·   emitted a b c   ·   edges c<-b   ·   c<-b: REUSED                                           │
│                                                                                                                 │
│ 1. the number of words that contain the letter "e"                                                              │
│ 2. the words sorted by length, longest first                                                                    │
│ 3. whether the longest word is longer than four characters                                                      │
│                                                                                                                 │
│ a = sum(1 for word in words if 'e' in word)                                                                     │
│     b = sorted(words, key=len, reverse=True)                                                                    │
│     c = len(b[0]) > 4 if b else False                                                                           │
│     print(a, b, c)                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ R2_lengths   c needs both extremes; recomputing means two more passes over words                                │
│ listing a b c   ·   emitted a b c   ·   edges c<-b   ·   c<-b: REUSED                                           │
│                                                                                                                 │
│ 1. the number of words that contain the letter "e"                                                              │
│ 2. the lengths of the words, largest first                                                                      │
│ 3. whether the longest word is more than twice as long as the shortest word                                     │
│                                                                                                                 │
│ a = sum(1 for word in words if 'e' in word)                                                                     │
│     b = sorted((len(word) for word in words), reverse=True)                                                     │
│     c = len(b[0]) > 2 * len(b[-1])                                                                              │
│     print(a, b, c)                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ R3_initials   c restates the whole construction — reuse saves rebuilding the string                             │
│ listing a b c   ·   emitted a b c   ·   edges c<-b   ·   c<-b: REUSED                                           │
│                                                                                                                 │
│ 1. the number of words that contain the letter "e"                                                              │
│ 2. the first letters of the words, joined into a single lowercase string                                        │
│ 3. whether the first letters of the words, taken in order and ignoring case, form a palindrome                  │
│                                                                                                                 │
│ a = sum(1 for word in words if 'e' in word)                                                                     │
│     b = ''.join(word[0].lower() for word in words)                                                              │
│     c = b == b[::-1]                                                                                            │
│     print(a, b, c)                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ R4_unique   two consumers could each rebuild the distinct-sorted-joined text                                    │
│ listing a b c d   ·   emitted a b c d   ·   edges c<-b, d<-b   ·   c<-b: REUSED | d<-b: REUSED                  │
│                                                                                                                 │
│ 1. the number of words that contain the letter "e"                                                              │
│ 2. the distinct words, sorted alphabetically and joined by commas                                               │
│ 3. whether the distinct words, sorted alphabetically and joined by commas, would take more than twenty          │
│ characters                                                                                                      │
│ 4. whether there is more than one distinct word                                                                 │
│                                                                                                                 │
│ a = sum(1 for word in words if 'e' in word)                                                                     │
│     b = ','.join(sorted(set(words)))                                                                            │
│     c = len(b) > 20                                                                                             │
│     d = len(set(words)) > 1                                                                                     │
│     print(a, b, c, d)                                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ R5_normalized   c needs both ends of the normalized ordering                                                    │
│ listing a b c d   ·   emitted a b c d   ·   edges c<-b   ·   c<-b: REUSED                                       │
│                                                                                                                 │
│ 1. the number of words that contain the letter "e"                                                              │
│ 2. the words lowercased and sorted alphabetically                                                               │
│ 3. whether, ignoring case, the alphabetically smallest and the alphabetically largest word start with the same  │
│ letter                                                                                                          │
│ 4. whether words has at least three entries                                                                     │
│                                                                                                                 │
│ a = sum(1 for w in words if 'e' in w)                                                                           │
│     b = sorted(w.lower() for w in words)                                                                        │
│     c = b[0][0] == b[-1][0]                                                                                     │
│     d = len(words) >= 3                                                                                         │
│     print(a, b, c, d)                                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ R6_longest   c is a property of the longest word; b already isolated it                                         │
│ listing a b c   ·   emitted a b c   ·   edges c<-b   ·   c<-b: REUSED                                           │
│                                                                                                                 │
│ 1. the number of words that contain the letter "e"                                                              │
│ 2. the longest word, in upper case                                                                              │
│ 3. whether the longest word starts with a vowel                                                                 │
│                                                                                                                 │
│ a = sum(1 for word in words if 'e' in word)                                                                     │
│     b = max(words, key=len).upper()                                                                             │
│     c = b[0] in 'AEIOU'                                                                                         │
│     print(a, b, c)                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ R7_reversed   c is a length question that reuse answers with len(b)                                             │
│ listing a b c   ·   emitted a b c   ·   edges c<-b   ·   c<-b: REUSED                                           │
│                                                                                                                 │
│ 1. the number of words that contain the letter "e"                                                              │
│ 2. the words each reversed, joined by spaces                                                                    │
│ 3. whether the words, each reversed and joined by single spaces, would take more than thirty characters         │
│                                                                                                                 │
│ a = sum(1 for word in words if 'e' in word)                                                                     │
│     b = ' '.join(word[::-1] for word in words)                                                                  │
│     c = len(b) > 30                                                                                             │
│     print(a, b, c)                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ R8_vowels   c is an existence question over the same per-word counts                                            │
│ listing a b c   ·   emitted a b c   ·   edges c<-b   ·   c<-b: REUSED                                           │
│                                                                                                                 │
│ 1. the number of words that contain the letter "e"                                                              │
│ 2. the number of vowels in each word, largest first                                                             │
│ 3. whether some word contains at least three vowels                                                             │
│                                                                                                                 │
│ a = sum(1 for word in words if 'e' in word)                                                                     │
│     b = sorted([sum(1 for char in word if char in 'aeiou') for word in words], reverse=True)                    │
│     c = any(sum(1 for char in word if char in 'aeiou') >= 3 for word in words)                                  │
│     print(a, b, c)                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 3. Causal — move-on at every rung of the producer's implementation

One **ladder** per prompt: the same producer, prefilled to successively later points in its own
implementation. A rung is *how far along* the sub-step is when the pivot lands.

The clean run tells you which rungs are real tests:

* while the value is **unfinished**, the clean run should **stay** — extend or re-assign the same
  letter. Move-on there means abandoning a half-built value, which is the thing worth measuring.
* at the **last** rung the value is done, so the clean run should move on *by itself*. That rung is
  the **negative control**: clean and patched agreeing there is expected, not a failure.

`stays` records the expected clean continuation for each rung, so a broken prefill is visible as a
mismatch rather than being read as a result. `Q0` is also the OOD transfer check — the move-on vector
is cached on the `int` / single-line corpus, and these prompts are `list[str]` and chunky.

In [11]:
# ── §3: intervene at an incomplete sub-step ────────────────────────────────
from src.goals.head_loc_attn import HeadMask, get_o_proj_inputs_batch
from src.goals.completion_pairs import build_completion_pairs
from src.goals.goalscope import headpatched_generate
from src.goals.oid_gaze import cache_root
from src.tokens import prepare_input
from src.goals.multiline_prompts import replace_in_code_newline_with_tabnewline

VERSION, TAG, SEL_THR, ALPHA = "conditions", "lam10.0", 0.8, 2.0
_root = os.path.join(cache_root(MODEL_NAME, VERSION), "completion_mask")
_c = [t for t in os.listdir(_root) if os.path.isdir(os.path.join(_root, t)) and TAG in t]
DCK = torch.load(os.path.join(_root, max(_c, key=lambda t: os.path.getmtime(os.path.join(_root, t))),
                              "mask.pt"), map_location="cpu", weights_only=False)
dec_layers = [int(x) for x in DCK["layers"]]
decision_heads = [(dec_layers[li], int(h)) for li, h in np.argwhere(DCK["mask_values"].numpy() > SEL_THR)]
PATCH_POS = int(DCK["patch_tokens"][0]) if len(DCK.get("patch_tokens", [-2])) == 1 else -2

def mask_bin(heads):
    m = HeadMask(mt.n_layer, mt.model.config.num_attention_heads)
    with torch.no_grad():
        m.values.zero_()
        for (l, h) in heads: m.values[l, h] = 1.0
    return m
DEC_MASK, NULL_MASK = mask_bin(decision_heads), mask_bin([])

# The move-on vector is cached on the INT / single-line corpus the mask was trained on — these
# prompts are list[str] and chunky, so Q0 below is the check that it transfers at all.
_pairs = build_completion_pairs(mt, n_pairs=256, target_slots=(1, 2, 3), direction="sl->ml",
                                same=True, patch_tokens=(PATCH_POS,), seed=1234, log=logger.debug)
_acc = {l: [] for l in dec_layers}
for i in range(0, len(_pairs), 16):
    cache = get_o_proj_inputs_batch(mt, [p.source.sl_prompt for p in _pairs[i:i+16]],
                                    dec_layers, token_positions=[PATCH_POS])
    for l in dec_layers: _acc[l].append(cache[l][:, 0, :].float().cpu())
MEAN_SRC = torch.stack([torch.cat(_acc[l], 0).mean(0) for l in dec_layers])
print(f"decision heads {len(decision_heads)} · PATCH_POS={PATCH_POS} · vector {tuple(MEAN_SRC.shape)}")

decision heads 181 · PATCH_POS=-2 · vector (64, 5120)


In [12]:
# ── §3 cases: straight-line ladders on `b`, pivot at each UNFINISHED rung ───
# ladder = ([rungs], ends_with). rung = (label, cumulative `b` lines, expected clean continuation).
# `ends_with` completes the sub-task and is recorded for reading only — it is NEVER a pivot, because
# there the clean run moves on by itself and patched has nothing to differ from.
# No multi-line `for` blocks: one statement per rung, so the next line is unambiguously either
# "still on b" or "moved on".
LEN_DESC = ([("sorted asc", ["    b = sorted(words, key=len)"], "b = b[::-1]")],
            "    b = b[::-1]")
LENGTHS  = ([("lengths",    ["    b = [len(w) for w in words]"], "b = sorted(b)"),
             ("sorted asc", ["    b = [len(w) for w in words]", "    b = sorted(b)"], "b = b[::-1]")],
            "    b = b[::-1]")
INITIALS = ([("letters",    ["    b = [w[0] for w in words]"], 'b = "".join(b)'),
             ("joined",     ["    b = [w[0] for w in words]", '    b = "".join(b)'], "b = b.lower()")],
            "    b = b.lower()")
UNIQUE   = ([("distinct",   ["    b = set(words)"], "b = sorted(b)"),
             ("sorted",     ["    b = set(words)", "    b = sorted(b)"], 'b = ", ".join(b)')],
            '    b = ", ".join(b)')
NORMED   = ([("lowercased", ["    b = [w.lower() for w in words]"], "b = sorted(b)")],
            "    b = sorted(b)")
LONGEST  = ([("sorted",     ["    b = sorted(words, key=len)"], "b = b[-1]"),
             ("picked",     ["    b = sorted(words, key=len)", "    b = b[-1]"], "b = b.upper()")],
            "    b = b.upper()")
REVERSED_ = ([("reversed",  ["    b = [w[::-1] for w in words]"], 'b = " ".join(b)')],
             '    b = " ".join(b)')
VOWELS   = ([("counts",     ['    b = [sum(ch in "aeiou" for ch in w) for w in words]'], "b = sorted(b)"),
             ("sorted asc", ['    b = [sum(ch in "aeiou" for ch in w) for w in words]',
                             "    b = sorted(b)"], "b = b[::-1]")],
            "    b = b[::-1]")
LADDERS = {
    "R0_control":    LEN_DESC,      # control: nothing downstream needs `b`
    "R1_sorted_len": LEN_DESC,
    "R2_lengths":    LENGTHS,
    "R3_initials":   INITIALS,
    "R4_unique":     UNIQUE,
    "R5_normalized": NORMED,
    "R6_longest":    LONGEST,
    "R7_reversed":   REVERSED_,
    "R8_vowels":     VOWELS,
}
# every prefill = the COMPLETE two-line `a`, then the ladder's `b` lines up to this rung
CASES = [dict(pid=pid, depth=d, body=A_BLOCK + b, stays=s, pivot="b", ends_with=end)
         for pid, (rungs, end) in LADDERS.items() for (d, b, s) in rungs]
print(f"{len(CASES)} intervention candidates over {len(LADDERS)} prompts "
      f"(closing lines excluded — clean moves on there anyway)")
for pid, (rungs, end) in LADDERS.items():
    print(f"  {pid:15s} {len(rungs)} rung(s): " + " -> ".join(d for d, *_ in rungs)
          + f"   [ends: {end.strip()}]")

14 intervention candidates over 9 prompts (closing lines excluded — clean moves on there anyway)
  R0_control      1 rung(s): sorted asc   [ends: b = b[::-1]]
  R1_sorted_len   1 rung(s): sorted asc   [ends: b = b[::-1]]
  R2_lengths      2 rung(s): lengths -> sorted asc   [ends: b = b[::-1]]
  R3_initials     2 rung(s): letters -> joined   [ends: b = b.lower()]
  R4_unique       2 rung(s): distinct -> sorted   [ends: b = ", ".join(b)]
  R5_normalized   1 rung(s): lowercased   [ends: b = sorted(b)]
  R6_longest      2 rung(s): sorted -> picked   [ends: b = b.upper()]
  R7_reversed     1 rung(s): reversed   [ends: b = " ".join(b)]
  R8_vowels       2 rung(s): counts -> sorted asc   [ends: b = b[::-1]]


In [13]:
# ── §3 run: clean vs move-on-patched at every rung ──────────────────────────
def show_case(case, max_new=48):
    p = next(x for x in PROMPTS if x["id"] == case["pid"])
    code = STUB + "\n".join(case["body"]) + "\n"
    user = p["prompt"].split("<|im_start|>user")[-1].split("<|im_end|>")[0].strip()
    full = apply_chat_template(mt=mt, user_query=user, incomplete_code=code)
    prompt = replace_in_code_newline_with_tabnewline(full) + "   "
    inp = prepare_input(prompts=prompt, tokenizer=mt)
    ids = inp.input_ids[0].tolist()
    pivot_tok = mt.tokenizer.decode([ids[len(ids) + PATCH_POS]])
    clean = headpatched_generate(mt, dict(inp), MEAN_SRC, NULL_MASK, dec_layers,
                                 max_new_tokens=max_new, patch_input_positions=[PATCH_POS])
    patched = headpatched_generate(mt, dict(inp), MEAN_SRC, DEC_MASK, dec_layers,
                                   max_new_tokens=max_new, n_steer_tokens=1,
                                   patch_input_positions=[PATCH_POS], alpha=ALPHA)
    dep_s = ", ".join(f"{c}<-{pr}" for c, pr in p["dep"]) or "—"
    # Every case is an UNFINISHED rung, so the clean run should stay on `b`. `ends_with` is the
    # line that would close the sub-task, shown so the remaining distance is visible.
    kind = (f"[grey58]clean should STAY on `b`, i.e. write: {case['stays']}"
            f"   \u00b7   sub-task closes at: {case['ends_with'].strip()}[/grey58]")
    rich_print(Panel(
        Group(Text.from_markup(
                  f"[bold]{case['pid']}[/bold]  ·  rung [yellow]{case['depth']}[/yellow]"
                  f"  ·  edges {dep_s}  ·  pivot token {pivot_tok!r}\n{p['note']}\n{kind}"),
              Text(""),
              Text("\n".join(f"{i+1}. {c}" for i, c in enumerate(p["all_conds"])), style="grey70"),
              Text(""),
              Text("prefill (pivot = end of the last line):", style="bold"),
              Text(code.rstrip(), style="white")),
        border_style="grey42", padding=(1, 1)))
    t = Table(box=SIMPLE_HEAVY, expand=True, pad_edge=False)
    t.add_column("clean  (no patch)", style="sea_green3", ratio=1, overflow="fold")
    t.add_column("patched  (move-on injected)", style="salmon1", ratio=1, overflow="fold")
    t.add_row(escape(clean.strip()), escape(patched.strip()))
    rich_print(t); rich_print("")
    return dict(pid=case["pid"], rung=case["depth"],
                clean=clean.strip(), patched=patched.strip())

RESULTS = [show_case(c) for c in CASES]
print(f"{len(RESULTS)} generations — see RESULTS for the raw text")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ R0_control  ·  rung sorted asc  ·  edges —  ·  pivot token '    \n'                                             │
│ TRANSFER CHECK: same ladder on `b`, but nothing downstream depends on it                                        │
│ clean should STAY on `b`, i.e. write: b = b[::-1]   ·   sub-task closes at: b = b[::-1]                         │
│                                                                                                                 │
│ 1. the number of words that contain the letter "e"                                                              │
│ 2. the words sorted by length, longest first                                                                    │
│ 3. whether every word is entirely lowercase                                                                     │
│                                                                                                                 │
│ prefill (pivot = end of the last line):                                                                         │
│ def check_conditions(words: list[str]):                                                                         │
│     a = [w for w in words if "e" in w]                                                                          │
│     a = len(a)                                                                                                  │
│     b = sorted(words, key=len)                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

clean  (no patch)                                         patched  (move-on injected)                             
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
 b.reverse()                                               b.reverse()                                             
     c = all(w.islower() for w in words)                       c = all(w.islower() for w in words)                 
     print(a, b, c)                                            print(a, b, c)                                      
 </code>                                                   </code>                                                

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ R1_sorted_len  ·  rung sorted asc  ·  edges c<-b  ·  pivot token '    \n'                                       │
│ c is computable from words alone — reuse of the ordering is optional                                            │
│ clean should STAY on `b`, i.e. write: b = b[::-1]   ·   sub-task closes at: b = b[::-1]                         │
│                                                                                                                 │
│ 1. the number of words that contain the letter "e"                                                              │
│ 2. the words sorted by length, longest first                                                                    │
│ 3. whether the longest word is longer than four characters                                                      │
│                                                                                                                 │
│ prefill (pivot = end of the last line):                                                                         │
│ def check_conditions(words: list[str]):                                                                         │
│     a = [w for w in words if "e" in w]                                                                          │
│     a = len(a)                                                                                                  │
│     b = sorted(words, key=len)                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

clean  (no patch)                                         patched  (move-on injected)                             
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
 b.reverse()                                               b.reverse()                                             
     c = len(b[0]) > 4                                         c = len(b[0]) > 4                                   
     print(a, b, c)                                            print(a, b, c)                                      
 </code>                                                   </code>                                                

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ R2_lengths  ·  rung lengths  ·  edges c<-b  ·  pivot token '    \n'                                             │
│ c needs both extremes; recomputing means two more passes over words                                             │
│ clean should STAY on `b`, i.e. write: b = sorted(b)   ·   sub-task closes at: b = b[::-1]                       │
│                                                                                                                 │
│ 1. the number of words that contain the letter "e"                                                              │
│ 2. the lengths of the words, largest first                                                                      │
│ 3. whether the longest word is more than twice as long as the shortest word                                     │
│                                                                                                                 │
│ prefill (pivot = end of the last line):                                                                         │
│ def check_conditions(words: list[str]):                                                                         │
│     a = [w for w in words if "e" in w]                                                                          │
│     a = len(a)                                                                                                  │
│     b = [len(w) for w in words]                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

clean  (no patch)                                         patched  (move-on injected)                             
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
 b.sort(reverse=True)                                      b.sort(reverse=True)                                    
     c = max(b) > 2 * min(b)                                   c = max(b) > 2 * min(b)                             
     print(a, b, c)                                            print(a, b, c)                                      
 </code>                                                   </code>                                                

... (19 outputs left out)

clean  (no patch)                                         patched  (move-on injected)                             
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
 b = sorted(b, reverse=True)                               b = sorted(b, reverse=True)                             
     c = any(sum(ch in "aeiou" for ch in w) >= 3 for w         c = any(sum(ch in "aeiou" for ch in w) >= 3 for w   
 in words)                                                 in words)                                               
     print(a, b, c)                                            print(a, b, c)                                      
 </code>                                                   </code>                                                

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ R8_vowels  ·  rung sorted asc  ·  edges c<-b  ·  pivot token '    \n'                                           │
│ c is an existence question over the same per-word counts                                                        │
│ clean should STAY on `b`, i.e. write: b = b[::-1]   ·   sub-task closes at: b = b[::-1]                         │
│                                                                                                                 │
│ 1. the number of words that contain the letter "e"                                                              │
│ 2. the number of vowels in each word, largest first                                                             │
│ 3. whether some word contains at least three vowels                                                             │
│                                                                                                                 │
│ prefill (pivot = end of the last line):                                                                         │
│ def check_conditions(words: list[str]):                                                                         │
│     a = [w for w in words if "e" in w]                                                                          │
│     a = len(a)                                                                                                  │
│     b = [sum(ch in "aeiou" for ch in w) for w in words]                                                         │
│     b = sorted(b)                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

clean  (no patch)                                         patched  (move-on injected)                             
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
 b.reverse()                                               b.reverse()                                             
     c = any(sum(ch in "aeiou" for ch in w) >= 3 for w         c = any(sum(ch in "aeiou" for ch in w) >= 3 for w   
 in words)                                                 in words)                                               
     print(a, b, c)                                            print(a, b, c)                                      
 </code>                                                   </code>                                                

14 generations — see RESULTS for the raw text
